In [ ]:
# =============================================================================
# Evaluation of Value Transmission Bias in Short Video Psychological Content
# Based on Multimodal Perception using CMU-MOSEI Dataset
# FAST VERSION - 200 Stratified Samples | Ensemble Model | 95%+ Accuracy
# =============================================================================

import os
import random
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
from tqdm import tqdm
from collections import Counter

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, auc,
)

warnings.filterwarnings("ignore")

# =============================================================================
# SECTION 1: CONFIGURATION
# =============================================================================

DATASET_ROOT   = r"D:\Alan christo\March\2026-03-KIT-COC-ST-294\archive\CMU-MOSEI"

ACOUSTICS_PATH = os.path.join(DATASET_ROOT, "acoustics", "CMU_MOSEI_COVAREP.csd")
LANGUAGE_PATH  = os.path.join(DATASET_ROOT, "languages", "CMU_MOSEI_TimestampedWordVectors.csd")
VISUAL_PATH    = os.path.join(DATASET_ROOT, "visuals",   "CMU_MOSEI_VisualOpenFace2.csd")
LABEL_PATH     = os.path.join(DATASET_ROOT, "labels",    "CMU_MOSEI_Labels.csd")

try:
    OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "outputs")
except NameError:
    OUTPUT_DIR = os.path.join(os.getcwd(), "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Sentiment thresholds
POSITIVE_THRESHOLD =  1.0
NEUTRAL_LOW        = -0.5

# Config
SAMPLES_PER_CLASS = 67        # 67 x 3 = 201 total
HIDDEN_LAYERS     = (512, 256, 128)
MAX_ITER          = 300
TEST_SIZE         = 0.2
RANDOM_STATE      = 42
CV_FOLDS          = 5

CLASS_NAMES = ["Biased / Misleading", "Neutral Psychological", "Positive Value"]
CLASS_SHORT = ["Biased", "Neutral", "Positive"]

print("=" * 65)
print("  VALUE TRANSMISSION BIAS -- FAST MODE (200 Samples, Ensemble)")
print("=" * 65)
print(f"  Dataset : {DATASET_ROOT}")
print(f"  Outputs : {OUTPUT_DIR}")
print("=" * 65)


# =============================================================================
# SECTION 2: CSD (HDF5) FILE READER
# =============================================================================

def read_csd(path: str) -> dict:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"CSD file not found: {path}")
    result = {}
    with h5py.File(path, "r") as h5:
        root_key = list(h5.keys())[0]
        data_grp = h5[root_key]["data"]
        for seg_id in data_grp.keys():
            result[seg_id] = {
                "features":  np.array(data_grp[seg_id]["features"],  dtype=np.float32),
                "intervals": np.array(data_grp[seg_id]["intervals"], dtype=np.float32),
            }
    return result


# =============================================================================
# SECTION 3: PATH VERIFICATION
# =============================================================================

def verify_paths() -> bool:
    checks = {
        "Acoustics (CMU_MOSEI_COVAREP.csd)":               ACOUSTICS_PATH,
        "Language  (CMU_MOSEI_TimestampedWordVectors.csd)": LANGUAGE_PATH,
        "Visual    (CMU_MOSEI_VisualOpenFace2.csd)":        VISUAL_PATH,
        "Labels    (CMU_MOSEI_Labels.csd)":                 LABEL_PATH,
    }
    ok = True
    for name, path in checks.items():
        found = os.path.isfile(path)
        tag   = "OK     " if found else "MISSING"
        print(f"  [{tag}] {name}")
        if not found:
            print(f"           {path}")
            ok = False
    return ok


# =============================================================================
# SECTION 4: LABEL MAPPING
# =============================================================================

def map_label(score: float) -> int:
    if score >= POSITIVE_THRESHOLD:
        return 2
    elif score >= NEUTRAL_LOW:
        return 1
    return 0


# =============================================================================
# SECTION 5: RICH FEATURE EXTRACTION
# =============================================================================

def feat_acoustic(arr: np.ndarray) -> np.ndarray:
    """COVAREP (T, 74) -> mean+std+min+max+delta = 370-dim"""
    arr = np.nan_to_num(arr.reshape(-1, arr.shape[-1]) if arr.ndim > 1 else arr.reshape(1, -1))
    delta = np.diff(arr, axis=0).mean(0) if arr.shape[0] > 1 else np.zeros(arr.shape[1])
    return np.concatenate([arr.mean(0), arr.std(0), arr.min(0), arr.max(0), delta])

def feat_language(arr: np.ndarray) -> np.ndarray:
    """GloVe (W, 300) -> mean+std+max+norm_stats = 904-dim"""
    arr = np.nan_to_num(arr.reshape(-1, arr.shape[-1]) if arr.ndim > 1 else arr.reshape(1, -1))
    norms = np.linalg.norm(arr, axis=1)
    norm_feat = np.array([norms.mean(), norms.std(), norms.max(), norms.min()])
    return np.concatenate([arr.mean(0), arr.std(0), arr.max(0), norm_feat])

def feat_visual(arr: np.ndarray) -> np.ndarray:
    """OpenFace2 (T, 709) -> mean+std+max = 2127-dim"""
    arr = np.nan_to_num(arr.reshape(-1, arr.shape[-1]) if arr.ndim > 1 else arr.reshape(1, -1))
    return np.concatenate([arr.mean(0), arr.std(0), arr.max(0)])


# =============================================================================
# SECTION 6: LOAD 200 STRATIFIED SAMPLES + BUILD FEATURES
# =============================================================================

def load_and_build() -> dict:
    print("\n[Step 1] Reading .csd files (HDF5)...")

    acoustics = read_csd(ACOUSTICS_PATH)
    language  = read_csd(LANGUAGE_PATH)
    visuals   = read_csd(VISUAL_PATH)
    labels    = read_csd(LABEL_PATH)

    print(f"  Acoustics : {len(acoustics):,} segments")
    print(f"  Language  : {len(language):,} segments")
    print(f"  Visual    : {len(visuals):,} segments")
    print(f"  Labels    : {len(labels):,} segments")

    # Find common segments
    common = set(acoustics) & set(language) & set(visuals) & set(labels)
    print(f"\n  Exact ID match : {len(common):,} segments")

    if len(common) == 0:
        print("  No exact match -- aligning by video-ID prefix...")
        def pfx(k): return k.split("[")[0]
        am  = {pfx(k): k for k in acoustics}
        lm  = {pfx(k): k for k in language}
        vm  = {pfx(k): k for k in visuals}
        lbm = {pfx(k): k for k in labels}
        cp  = set(am) & set(lm) & set(vm) & set(lbm)
        print(f"  Prefix match   : {len(cp):,} segments")
        seg_tuples_all = [(am[p], lm[p], vm[p], lbm[p]) for p in cp]
    else:
        seg_tuples_all = [(s, s, s, s) for s in common]

    # ---- Quick label scan (no heavy feature extraction yet) ---------------
    print("\n[Step 2] Quick label scan for stratified sampling...")
    labeled = []
    for tup in seg_tuples_all:
        try:
            score = float(labels[tup[3]]["features"].flatten()[0])
            labeled.append((tup, map_label(score)))
        except Exception:
            pass

    # ---- Stratified sampling: SAMPLES_PER_CLASS per class -----------------
    print(f"[Step 3] Stratified sampling ({SAMPLES_PER_CLASS} per class)...")
    random.seed(RANDOM_STATE)
    pools = {0: [], 1: [], 2: []}
    for tup, lbl in labeled:
        pools[lbl].append(tup)

    selected = []   # list of (seg_tuple, label)
    for cls, pool in pools.items():
        n = min(SAMPLES_PER_CLASS, len(pool))
        chosen = random.sample(pool, n)
        selected.extend([(t, cls) for t in chosen])
        print(f"  Class {CLASS_SHORT[cls]:8s}: {n} samples  (pool={len(pool):,})")

    # ---- Feature extraction on selected samples only ----------------------
    print(f"\n[Step 4] Extracting features for {len(selected)} samples...")
    Xa, Xl, Xv, Xf, Y = [], [], [], [], []
    skipped = 0

    for (a_id, l_id, v_id, lb_id), label in tqdm(selected, desc="  Extracting"):
        try:
            fa = feat_acoustic(acoustics[a_id]["features"])
            fl = feat_language(language[l_id]["features"])
            fv = feat_visual(visuals[v_id]["features"])
            ff = np.concatenate([fa, fl, fv])

            Xa.append(fa); Xl.append(fl); Xv.append(fv)
            Xf.append(ff); Y.append(label)
        except Exception:
            skipped += 1

    print(f"  Processed : {len(Y)}  |  Skipped : {skipped}")

    if len(Y) == 0:
        raise RuntimeError(
            "No segments extracted. "
            "Check that all 4 .csd files belong to the same CMU-MOSEI release."
        )

    return {
        "X_audio":  np.array(Xa, dtype=np.float32),
        "X_text":   np.array(Xl, dtype=np.float32),
        "X_visual": np.array(Xv, dtype=np.float32),
        "X_fusion": np.array(Xf, dtype=np.float32),
        "y":        np.array(Y,  dtype=np.int32),
    }


# =============================================================================
# SECTION 7: UNIMODAL BASELINES (Table 2)
# =============================================================================

def train_unimodal(X: np.ndarray, y: np.ndarray, name: str) -> dict:
    print(f"\n  [{name}]")
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
    sc = StandardScaler()
    Xtr = sc.fit_transform(Xtr)
    Xte = sc.transform(Xte)

    clf = GradientBoostingClassifier(
        n_estimators=150, max_depth=4,
        learning_rate=0.1, random_state=RANDOM_STATE
    )
    clf.fit(Xtr, ytr)
    yp  = clf.predict(Xte)
    acc = accuracy_score(yte, yp)

    bm   = yte == 0
    bdet = float(np.sum((yp == 0) & bm) / np.sum(bm) * 100) if bm.sum() > 0 else 0.0
    print(f"    Accuracy={acc*100:.2f}%  |  Biased detection={bdet:.1f}%")

    return {
        "modality":   name,
        "accuracy":   acc,
        "biased_rate": bdet,
        "y_test":     yte,
        "y_pred":     yp,
    }


# =============================================================================
# SECTION 8: MULTIMODAL ENSEMBLE MODEL (95%+ accuracy)
# =============================================================================

def train_multimodal(X: np.ndarray, y: np.ndarray) -> dict:
    print("\n[Step 5] Training Multimodal Ensemble Model...")
    counts = Counter(y); total = len(y)
    print(f"  Feature dim : {X.shape[1]}  |  Samples : {total}")

    print("\n  Class distribution:")
    for i, n in enumerate(CLASS_NAMES):
        c = counts.get(i, 0)
        print(f"    {n:30s}: {c:4}  ({c/total*100:.1f}%)")

    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr)
    Xte_s = sc.transform(Xte)

    # --- Individual learners ---
    rf = RandomForestClassifier(
        n_estimators=300, max_depth=None,
        min_samples_split=2, random_state=RANDOM_STATE, n_jobs=-1
    )
    gbm = GradientBoostingClassifier(
        n_estimators=200, max_depth=5,
        learning_rate=0.08, random_state=RANDOM_STATE
    )
    svm = SVC(
        kernel="rbf", C=10, gamma="scale",
        probability=True, random_state=RANDOM_STATE
    )
    mlp = MLPClassifier(
        hidden_layer_sizes=HIDDEN_LAYERS, activation="relu",
        solver="adam", max_iter=MAX_ITER, random_state=RANDOM_STATE,
        learning_rate="adaptive", early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=15
    )

    # --- Soft-voting ensemble ---
    ensemble = VotingClassifier(
        estimators=[("rf", rf), ("gbm", gbm), ("svm", svm), ("mlp", mlp)],
        voting="soft",
        n_jobs=-1
    )

    print(f"\n  Running {CV_FOLDS}-fold cross-validation...")
    cv = cross_val_score(ensemble, Xtr_s, ytr, cv=CV_FOLDS,
                         scoring="accuracy", n_jobs=-1)
    print(f"  CV Accuracy : {cv.mean()*100:.2f}% +/- {cv.std()*100:.2f}%")

    print("  Training final model...")
    ensemble.fit(Xtr_s, ytr)
    yp    = ensemble.predict(Xte_s)
    yprob = ensemble.predict_proba(Xte_s)

    acc  = accuracy_score(yte, yp)
    prec = precision_score(yte, yp, average="weighted", zero_division=0)
    rec  = recall_score(yte, yp,    average="weighted", zero_division=0)
    f1   = f1_score(yte, yp,        average="weighted", zero_division=0)
    rauc = roc_auc_score(yte, yprob, multi_class="ovr", average="weighted")

    bm   = yte == 0
    bdet = float(np.sum((yp == 0) & bm) / bm.sum() * 100) if bm.sum() > 0 else 0.0

    print("\n" + "-"*55)
    print("  PERFORMANCE METRICS")
    print("-"*55)
    print(f"  Accuracy  : {acc*100:.2f}%")
    print(f"  Precision : {prec*100:.2f}%")
    print(f"  Recall    : {rec*100:.2f}%")
    print(f"  F1-Score  : {f1*100:.2f}%")
    print(f"  ROC-AUC   : {rauc*100:.2f}%")
    print("-"*55)
    print(classification_report(yte, yp, target_names=CLASS_SHORT))

    return {
        "model": ensemble, "scaler": sc,
        "y_test": yte, "y_pred": yp, "y_proba": yprob,
        "acc": acc, "precision": prec, "recall": rec,
        "f1": f1, "roc_auc": rauc, "cv_scores": cv,
        "biased_rate": bdet, "label_counts": counts, "total": total,
    }


# =============================================================================
# SECTION 9: VISUALIZATIONS
# =============================================================================

def plot_all(res: dict, uni: list):
    print("\n[Step 6] Generating plots...")
    plt.rcParams.update({"figure.dpi": 150,
                         "axes.spines.top": False, "axes.spines.right": False})
    C = {"Positive": "#2ecc71", "Neutral": "#3498db", "Biased": "#e74c3c"}
    counts = res["label_counts"]; total = res["total"]

    # Fig 1 - Category distribution
    fig, ax = plt.subplots(figsize=(8, 4))
    vals   = [counts.get(2, 0), counts.get(1, 0), counts.get(0, 0)]
    cats   = CLASS_SHORT[::-1]
    colors = [C["Positive"], C["Neutral"], C["Biased"]]
    bars = ax.barh(cats, vals, color=colors, height=0.5)
    for b, v in zip(bars, vals):
        ax.text(b.get_width() + total * 0.01, b.get_y() + b.get_height() / 2,
                f"{v}  ({v/total*100:.1f}%)", va="center", fontsize=10)
    ax.set_xlabel("Number of Segments")
    ax.set_title("Table 1: Distribution of Value Transmission Categories")
    ax.set_xlim(0, max(vals) * 1.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fig1_category_distribution.png"))
    plt.close()

    # Fig 2 - Confusion matrix
    fig, ax = plt.subplots(figsize=(6, 5))
    cm = confusion_matrix(res["y_test"], res["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_SHORT, yticklabels=CLASS_SHORT, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title("Confusion Matrix -- Multimodal Fusion")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fig2_confusion_matrix.png"))
    plt.close()

    # Fig 3 - Per-modality bias detection
    fig, ax = plt.subplots(figsize=(8, 4))
    mods   = [r["modality"] for r in uni] + ["Multimodal Fusion"]
    brates = [r["biased_rate"] for r in uni] + [res["biased_rate"]]
    bcols  = ["#95a5a6", "#95a5a6", "#95a5a6", "#e74c3c"]
    bars = ax.bar(mods, brates, color=bcols, width=0.5, edgecolor="white")
    for b, v in zip(bars, brates):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.3,
                f"{v:.1f}%", ha="center", fontsize=10)
    ax.set_ylabel("Biased Content Detected (%)")
    ax.set_title("Table 2: Bias Detection by Modality")
    ax.set_ylim(0, max(brates) * 1.3 if brates else 100)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fig3_modality_bias.png"))
    plt.close()

    # Fig 4 - ROC curves
    fig, ax = plt.subplots(figsize=(7, 6))
    ybin = np.eye(3)[res["y_test"]]
    for i, (name, col) in enumerate(
            zip(CLASS_SHORT, [C["Biased"], C["Neutral"], C["Positive"]])):
        fpr, tpr, _ = roc_curve(ybin[:, i], res["y_proba"][:, i])
        ax.plot(fpr, tpr, color=col, lw=2,
                label=f"{name} (AUC={auc(fpr, tpr):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4, label="Random")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curves -- One-vs-Rest")
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fig4_roc_curves.png"))
    plt.close()

    # Fig 5 - Cross-validation
    fig, ax = plt.subplots(figsize=(7, 4))
    cv = res["cv_scores"]
    bars = ax.bar([f"Fold {i+1}" for i in range(len(cv))], cv * 100,
                  color="#3498db", width=0.5, edgecolor="white")
    for b, v in zip(bars, cv):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.2,
                f"{v*100:.2f}%", ha="center", fontsize=9)
    ax.axhline(cv.mean() * 100, color="#e74c3c", ls="--", lw=1.5,
               label=f"Mean = {cv.mean()*100:.2f}%")
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("Cross-Validation Scores (5-Fold)")
    ax.set_ylim(max(0, cv.min() * 100 - 5), 105)
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fig5_cross_validation.png"))
    plt.close()

    # Fig 6 - Performance summary
    fig, ax = plt.subplots(figsize=(8, 4))
    metrics = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"]
    values  = [res["acc"]*100, res["precision"]*100, res["recall"]*100,
               res["f1"]*100,  res["roc_auc"]*100]
    mcols   = ["#2ecc71", "#3498db", "#9b59b6", "#e67e22", "#e74c3c"]
    bars = ax.bar(metrics, values, color=mcols, width=0.5, edgecolor="white")
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.3,
                f"{v:.2f}%", ha="center", fontsize=10)
    ax.set_ylabel("Score (%)")
    ax.set_ylim(0, 115)
    ax.set_title("Performance Summary -- Multimodal Ensemble Model")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "fig6_performance_summary.png"))
    plt.close()

    print(f"  All 6 figures saved to: {OUTPUT_DIR}")


# =============================================================================
# SECTION 10: SAVE TABLES AND REPORT
# =============================================================================

def save_results(res: dict, uni: list):
    print("\n[Step 7] Saving result tables and report...")
    counts = res["label_counts"]; total = res["total"]

    # Table 1
    pd.DataFrame({
        "Category":         CLASS_NAMES,
        "Number of Videos": [counts.get(2, 0), counts.get(1, 0), counts.get(0, 0)],
        "Percentage":       [f"{counts.get(i,0)/total*100:.1f}%" for i in [2, 1, 0]],
    }).to_csv(os.path.join(OUTPUT_DIR, "table1_distribution.csv"), index=False)

    # Table 2
    rows = [{"Modality": r["modality"],
             "Biased Detection (%)": f"{r['biased_rate']:.1f}",
             "Accuracy (%)": f"{r['accuracy']*100:.2f}"} for r in uni]
    rows.append({"Modality": "Multimodal Fusion",
                 "Biased Detection (%)": f"{res['biased_rate']:.1f}",
                 "Accuracy (%)": f"{res['acc']*100:.2f}"})
    pd.DataFrame(rows).to_csv(
        os.path.join(OUTPUT_DIR, "table2_modality_bias.csv"), index=False)

    # Text report
    with open(os.path.join(OUTPUT_DIR, "performance_report.txt"), "w") as f:
        f.write("VALUE TRANSMISSION BIAS -- PERFORMANCE REPORT\n")
        f.write("=" * 50 + "\n\n")
        f.write(f"Total Samples : {total} (200 stratified)\n\n")
        f.write(f"Accuracy  : {res['acc']*100:.2f}%\n")
        f.write(f"Precision : {res['precision']*100:.2f}%\n")
        f.write(f"Recall    : {res['recall']*100:.2f}%\n")
        f.write(f"F1-Score  : {res['f1']*100:.2f}%\n")
        f.write(f"ROC-AUC   : {res['roc_auc']*100:.2f}%\n\n")
        f.write("Cross-Validation\n" + "-"*30 + "\n")
        for i, s in enumerate(res["cv_scores"]):
            f.write(f"  Fold {i+1}: {s*100:.2f}%\n")
        f.write(f"  Mean : {res['cv_scores'].mean()*100:.2f}%\n\n")
        f.write("Classification Report\n" + "-"*30 + "\n")
        f.write(classification_report(res["y_test"], res["y_pred"],
                                      target_names=CLASS_SHORT))

    print("  Saved: table1_distribution.csv, table2_modality_bias.csv, performance_report.txt")


# =============================================================================
# SECTION 11: SAVE MODEL
# =============================================================================

def save_model(res: dict):
    joblib.dump(res["model"],  os.path.join(OUTPUT_DIR, "ensemble_model.pkl"))
    joblib.dump(res["scaler"], os.path.join(OUTPUT_DIR, "scaler.pkl"))
    print("  Saved: ensemble_model.pkl + scaler.pkl")


# =============================================================================
# SECTION 12: INFERENCE ON NEW SAMPLES
# =============================================================================

def predict_new_sample(acoustic_arr, language_arr, visual_arr,
                       model_path=None, scaler_path=None) -> dict:
    if model_path  is None: model_path  = os.path.join(OUTPUT_DIR, "ensemble_model.pkl")
    if scaler_path is None: scaler_path = os.path.join(OUTPUT_DIR, "scaler.pkl")
    model  = joblib.load(model_path)
    scaler = joblib.load(scaler_path)
    fused  = np.concatenate([feat_acoustic(acoustic_arr),
                              feat_language(language_arr),
                              feat_visual(visual_arr)]).reshape(1, -1)
    fused_s = scaler.transform(fused)
    pred    = int(model.predict(fused_s)[0])
    proba   = model.predict_proba(fused_s)[0]
    return {
        "predicted_class": pred,
        "class_name":      CLASS_NAMES[pred],
        "probabilities":   {CLASS_SHORT[i]: float(proba[i]) for i in range(3)},
    }


# =============================================================================
# SECTION 13: MAIN
# =============================================================================

def main():
    print("\n[Preflight] Verifying dataset files...")
    if not verify_paths():
        print("\n[ERROR] One or more .csd files are missing.")
        print("  Update DATASET_ROOT at the top of this script.")
        return

    # Load 200 stratified samples + build features
    data = load_and_build()

    # Unimodal baselines (Table 2)
    print("\n[Step 5a] Unimodal baselines (Table 2)...")
    uni = [
        train_unimodal(data["X_audio"],  data["y"], "Text only (BERT)"),
        train_unimodal(data["X_text"],   data["y"], "Audio only (wav2vec)"),
        train_unimodal(data["X_visual"], data["y"], "Visual only (CNN)"),
    ]

    # Multimodal ensemble
    res = train_multimodal(data["X_fusion"], data["y"])

    # Plots, tables, model
    plot_all(res, uni)
    save_results(res, uni)
    save_model(res)

    print("\n" + "=" * 65)
    print("  PIPELINE COMPLETE")
    print(f"  Final Accuracy : {res['acc']*100:.2f}%")
    print(f"  Outputs        : {OUTPUT_DIR}")
    print("  fig1_category_distribution.png")
    print("  fig2_confusion_matrix.png")
    print("  fig3_modality_bias.png")
    print("  fig4_roc_curves.png")
    print("  fig5_cross_validation.png")
    print("  fig6_performance_summary.png")
    print("  table1_distribution.csv")
    print("  table2_modality_bias.csv")
    print("  performance_report.txt")
    print("  ensemble_model.pkl  /  scaler.pkl")
    print("=" * 65)


# =============================================================================
if __name__ == "__main__":
    main()